In [ ]:
COLAB_RUNTIME = True # Set to true to set up the experiment for a Google Colab runtime

In [ ]:
from pathlib import Path

def clone_repo(
    repo_url: str,
    repo_root: Path,
    git_ref: str = "main"
) -> None:
    """Clone the repository into the Google Colab VM.

    Args:
        repo_url (str): The URL of the repository to clone.
        repo_root (Path): The root directory for installing the repository.
        git_ref (str): The branch or tag to checkout.
        force_clone (bool): If True, force the clone even if the repository already exists.
    """
    %cd {str(os.path.basename(repo_root))}
    if not repo_root.exists():
        !git clone {repo_url} {str(repo_root)}
    else:
        print("Repo already exists at:", repo_root)

    %cd {str(repo_root)}
    !git fetch --all -q
    !git reset --hard origin/{git_ref}

    assert (repo_root / "utility_analysis").exists(), (
        f"Expected utility_analysis/ under {repo_root}. "
    )

if COLAB_RUNTIME:
    REPO_URL = "https://github.com/thowell332/llm-preferences.git"
    REPO_ROOT = Path( "/content/llm_preferences").expanduser()

    clone_repo(REPO_URL, REPO_ROOT)

In [ ]:
%pip install --upgrade pip
%pip install -r /content/llm_preferences/requirements.txt "jedi>=0.16" "rich<14"

In [ ]:
import os
import zipfile
from google.colab import drive
drive.mount('/content/drive')

!rsync -progress /content/drive/MyDrive/llama-32-1b-instruct-quantized.zip /content/tmp/llama-32-1b-instruct-quantized.zip
!unzip /content/tmp/llama-32-1b-instruct-quantized.zip -d /content/models/llama-32-1b-instruct-quantized

In [ ]:
MODEL_KEY = "llama-32-1b-instruct"
MODELS_PATH = REPO_ROOT / "utility_analysis" / "models.yaml"

In [ ]:
# --- Pilot: edit these ---
import sys
from pathlib import Path

import yaml
from transformers import AutoConfig
from IPython.display import display

import importlib
import notebook_runs
importlib.reload(notebook_runs)

%matplotlib inline

if "MODEL_KEY" not in dir() or "MODELS_PATH" not in dir():
    raise RuntimeError("Run the model setup cell first (MODEL_KEY, MODELS_PATH).")

NOTEBOOK_REPO_ROOT = REPO_ROOT if COLAB_RUNTIME else MODELS_PATH.parent.parent
LP_DIR = NOTEBOOK_REPO_ROOT / "utility_analysis" / "experiments" / "linear_probes"
if str(LP_DIR) not in sys.path:
    sys.path.insert(0, str(LP_DIR))

from notebook_runs import (
    run_forced_choice_dual_probe_pilot
)

fig, info = run_forced_choice_dual_probe_pilot(
    repo_root=repo_root,
    model_key=MODEL_KEY,
    save_dir=f"results_linear_probes_forced_choice/{MODEL_KEY}",
    save_suffix="pilot_forced_choice_single_role",
    options_path="../../shared_options/options.json",
    utilities_path=f"../../shared_utilities/options_custom/{MODEL_KEY}/results_utilities_{MODEL_KEY}_helpful_assistant.json",
    roles="helpful assistant",
    layers="all",
    backend="vllm",
    probe_mode="all",
    primary_metric="mse",
)
